In [1]:
import os
import tarfile
import glob
import shutil
from landsatxplore.api import API
from landsatxplore.earthexplorer import EarthExplorer
import rasterio
from rasterio.enums import Resampling
import numpy as np
from tqdm import tqdm
from shapely.geometry import box
import math

# ---------- CONFIG ----------
USERNAME = "Devgoyal0808"
PASSWORD = "Chocoholic@1"
OUTPUT_DIR = "landsat_downloads"
PATCH_OUTPUT_DIR = "landsat_patches"
REGIONS = [
    {"name": "Thar_Desert", "lat": 27.0, "lon": 71.0},
    {"name": "Sahara", "lat": 23.0, "lon": 13.0},
    {"name": "California", "lat": 38.0, "lon": -120.0},
    {"name": "Australia", "lat": -33.0, "lon": 149.0},
]
START_DATE = "2022-01-01"
END_DATE = "2023-12-31"
MAX_CLOUD = 20
SCENES_PER_REGION = 40
PATCH_SIZE_PIXELS = 128
STRIDE = 64  # overlap between patches
SCALE = 30  # meters per pixel for output patches
# ----------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PATCH_OUTPUT_DIR, exist_ok=True)

api = API(USERNAME, PASSWORD)
ee = EarthExplorer(USERNAME, PASSWORD)

def search_scenes(lat, lon):
    scenes = api.search(
        dataset="landsat_ot_c2_l2",
        latitude=lat,
        longitude=lon,
        start_date=START_DATE,
        end_date=END_DATE,
        max_cloud_cover=MAX_CLOUD
    )
    return scenes

def download_scene(entity_id, out_dir):
    try:
        ee.download(entity_id, output_dir=out_dir)
        return True
    except Exception as e:
        print("Download failed:", e)
        return False

def extract_tar_gz(tar_path, extract_to):
    try:
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(path=extract_to)
        return True
    except Exception as e:
        print("Extract failed:", e)
        return False

def find_band_files(scene_folder):
    # finds SR_B4 (red), SR_B5 (nir), ST_B10 (surface temp)
    sr_b4 = glob.glob(os.path.join(scene_folder, "*SR_B4*.TIF")) + glob.glob(os.path.join(scene_folder, "*_SR_B4.TIF"))
    sr_b5 = glob.glob(os.path.join(scene_folder, "*SR_B5*.TIF")) + glob.glob(os.path.join(scene_folder, "*_SR_B5.TIF"))
    st_b10 = glob.glob(os.path.join(scene_folder, "*ST_B10*.TIF")) + glob.glob(os.path.join(scene_folder, "*_ST_B10.TIF"))
    # Some file naming variations exist; return first matches
    return (sr_b4[0] if sr_b4 else None, sr_b5[0] if sr_b5 else None, st_b10[0] if st_b10 else None)

def read_band_as_float(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)
        profile = src.profile
    return arr, profile

def resample_to_match(src_arr, src_profile, target_profile, resampling=Resampling.bilinear):
    # Resample src to target using rasterio
    dst_transform = target_profile['transform']
    dst_width = target_profile['width']
    dst_height = target_profile['height']
    dst = np.empty((dst_height, dst_width), dtype=np.float32)
    with rasterio.MemoryFile() as memfile:
        with memfile.open(**src_profile) as dataset:
            dataset.write(src_arr, 1)
            dataset.close()
        with memfile.open() as dataset:
            rasterio.warp.reproject(
                source=rasterio.band(dataset, 1),
                destination=dst,
                src_transform=src_profile['transform'],
                src_crs=src_profile['crs'],
                dst_transform=dst_transform,
                dst_crs=target_profile['crs'],
                resampling=resampling
            )
    return dst

def compute_ndvi(red, nir):
    ndvi = (nir - red) / (nir + red + 1e-8)
    return ndvi

def compute_lst_from_st_band(st_arr, profile):
    # Attempt to infer if ST band is already surface temp in Kelvin.
    # Heuristic: if values between 200 and 350 -> likely Kelvin. If values > 1000, maybe scaled
    arr = st_arr.copy()
    valid = arr[np.isfinite(arr)]
    if valid.size == 0:
        return None
    vmin, vmax = np.percentile(valid, [2,98])
    # If values are reasonable for Kelvin
    if 200 < vmin < 350 and 200 < vmax < 350:
        lst_k = arr
    else:
        # Try known scale: sometimes thermal is stored as DN that require radiance conversion.
        # Many Collection2 Level-2 ST_B10 should be in Kelvin, but if not we try common scale factors:
        # Apply common multiplicative scale factor if data looks like large DN values
        # Caution: this is heuristic — validate visually on sample
        scale_guess = 0.00341802  # commonly used ML for brightness
        offset_guess = 150.0  # a possible additive offset sometimes used. This is a guess.
        lst_k = arr * scale_guess + offset_guess
        print("Applied heuristic scaling to ST band because values were outside [200,350]. Check output!")
    lst_c = lst_k - 273.15
    return lst_k, lst_c

def save_patch(output_path, array, profile, dtype=rasterio.float32):
    new_profile = profile.copy()
    new_profile.update({
        "count": 1,
        "dtype": dtype
    })
    with rasterio.open(output_path, 'w', **new_profile) as dst:
        dst.write(array.astype(dtype), 1)

def extract_patches_for_scene(red_path, nir_path, st_path, scene_folder, out_dir, patch_size=PATCH_SIZE_PIXELS, stride=STRIDE):
    # Read bands
    red, red_prof = read_band_as_float(red_path)
    nir, nir_prof = read_band_as_float(nir_path)
    st, st_prof = read_band_as_float(st_path)

    # If ST resolution differs (e.g., thermal at coarser res), resample ST to match red profile
    if (st_prof['transform'] != red_prof['transform']) or (st.shape != red.shape):
        print("Resampling thermal to optical grid...")
        st_resampled = resample_to_match(st, st_prof, red_prof, Resampling.bilinear)
    else:
        st_resampled = st

    ndvi = compute_ndvi(red, nir)
    lst_k_arr, lst_c_arr = compute_lst_from_st_band(st_resampled, st_prof)

    height, width = red.shape
    # Patch extraction
    patch_id = 0
    for y in range(0, height - patch_size + 1, stride):
        for x in range(0, width - patch_size + 1, stride):
            red_patch = red[y:y+patch_size, x:x+patch_size]
            nir_patch = nir[y:y+patch_size, x:x+patch_size]
            ndvi_patch = ndvi[y:y+patch_size, x:x+patch_size]
            lst_patch = lst_c_arr[y:y+patch_size, x:x+patch_size]  # Celsius
            # Filter empty/invalid patches
            if np.isnan(ndvi_patch).all():
                continue
            # Save multi-layer patch as numpy or geotiff per convenience
            patch_folder = os.path.join(out_dir, os.path.basename(scene_folder))
            os.makedirs(patch_folder, exist_ok=True)
            np.savez_compressed(
                os.path.join(patch_folder, f"patch_{patch_id:05d}.npz"),
                RED=red_patch, NIR=nir_patch, NDVI=ndvi_patch, LST_C=lst_patch
            )
            patch_id += 1
    return patch_id

# MAIN: search download and process
for region in REGIONS:
    print("Region:", region['name'])
    scenes = search_scenes(region['lat'], region['lon'])
    print("Found scenes:", len(scenes))
    downloaded = 0
    for scene in scenes[:SCENES_PER_REGION]:
        entity = scene['entity_id']
        print("Downloading", entity)
        out_dir = os.path.join(OUTPUT_DIR, region['name'])
        os.makedirs(out_dir, exist_ok=True)
        try:
            success = download_scene(entity, out_dir)
            if not success:
                continue
            # Find the downloaded tar.gz file in out_dir
            tarfiles = glob.glob(os.path.join(out_dir, "*.tar.gz"))
            if not tarfiles:
                print("No tar.gz found for scene", entity)
                continue
            tarfile_path = tarfiles[-1]  # pick latest
            extract_dir = os.path.join(out_dir, os.path.splitext(os.path.basename(tarfile_path))[0])
            os.makedirs(extract_dir, exist_ok=True)
            extract_tar_gz(tarfile_path, extract_dir)
            # Find band files
            red_path, nir_path, st_path = find_band_files(extract_dir)
            if not (red_path and nir_path and st_path):
                print("Missing band(s) in scene; skipping.")
                continue
            # Create patch output subdir
            patches_out = os.path.join(PATCH_OUTPUT_DIR, region['name'])
            os.makedirs(patches_out, exist_ok=True)
            num_patches = extract_patches_for_scene(red_path, nir_path, st_path, extract_dir, patches_out)
            print(f"Extracted {num_patches} patches for scene {entity}")
            downloaded += 1
            # Optionally cleanup extracted folder/tar if you want to save space
            # shutil.rmtree(extract_dir)
        except Exception as e:
            print("Scene processing failed:", e)
    print(f"Completed region {region['name']}: downloaded & processed {downloaded} scenes.")

# Logout
api.logout()
ee.logout()


USGSError: NOT_FOUND: Invalid Endpoint.